# 02 - LangGraph Workflow：建立多步驟 LLM 工作流

本筆記本示範如何使用 LangGraph `StateGraph` API 建構 workflow，
並在 node 中使用 `LLMService.call_llm()` 呼叫 LLM。

## 核心概念

直接使用 LangGraph 原生 API 建構 workflow，在 node 中呼叫 `LLMService.call_llm()` 處理 LLM 呼叫：

- **`LLMService`**：統一 LLM 呼叫入口（prompt 組裝 + litellm.completion）
- **`StateGraph`**：LangGraph 原生 API，定義 node 和 edge
- **`MessagesState`**：LangGraph 內建的 messages state

你可以自行擴展 state 加入所需欄位（如 `input_data`、`image` 等）。

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from app.utils.config import init_config
from app.logger import init_mlflow
from llm_service import LLMService
from langgraph.graph import END, START, StateGraph, MessagesState

cfg = init_config()
init_mlflow(cfg)

# 建立 LLMService
service = LLMService()

print("設定與 MLflow 初始化完成。")

## State 結構

直接使用 LangGraph 的 `MessagesState`（只有 `messages` 欄位）。

需要額外欄位時，自行擴展：

```python
from langgraph.graph import MessagesState
from typing import Any

class MyState(MessagesState):
    input_data: str = ""
    image_base64: str = ""
    result: dict[str, Any] = {}
```

## 範例一：最簡單的 LLM 呼叫

使用 `LLMService.call_llm()` 在 LangGraph node 中呼叫 LLM。

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# 定義 LLM node — 在 node 中呼叫 service.call_llm()
def call_llm_node(state: dict) -> dict:
    """從 messages 取得最後一則 user message，呼叫 LLM。"""
    messages = state.get("messages", [])

    # 取得最後一則 user message
    user_text = ""
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            user_text = msg.content
            break
        elif isinstance(msg, tuple) and msg[0] == "user":
            user_text = msg[1]
            break

    response = service.call_llm(
        user_prompt=user_text,
        system_prompt="你是一個有用的中文助手。",
    )
    return {"messages": [AIMessage(content=response.content)]}

# 建構最簡單的 workflow
graph = StateGraph(MessagesState)
graph.add_node("call_llm", call_llm_node)
graph.set_entry_point("call_llm")
graph.add_edge("call_llm", END)
simple_graph = graph.compile()

# 執行
try:
    result = simple_graph.invoke({"messages": [("user", "什麼是 LangGraph？")]})
    print("=== Workflow 結果 ===")
    print(f"最後回應: {result['messages'][-1].content[:200]}")
except Exception as e:
    print(f"[警告] 執行失敗：{e}")
    print("請確認 LLM 服務已啟動。")

## 範例二：自定義多步驟 Workflow

直接使用 LangGraph `StateGraph` API 建構自定義 workflow：

```
preprocess → call_llm → postprocess → END
```

In [ ]:
# 定義自定義 node functions
def preprocess(state: dict) -> dict:
    """前處理：擷取 user message 並加入分析指令。"""
    messages = state.get("messages", [])
    user_text = ""
    for msg in reversed(messages):
        if isinstance(msg, dict) and msg.get("role") == "user":
            user_text = msg["content"]
            break
        elif hasattr(msg, "type") and msg.type == "human":
            user_text = msg.content
            break

    print(f"[preprocess] 擷取輸入: {user_text[:50]}...")

    enhanced_prompt = f"請針對以下內容進行分析並提供摘要：\n\n{user_text}"
    return {
        "messages": [("user", enhanced_prompt)],
    }


def postprocess(state: dict) -> dict:
    """後處理：印出統計資訊。"""
    messages = state.get("messages", [])
    last = messages[-1] if messages else None
    response_text = last.content if hasattr(last, "content") else str(last)
    print(f"[postprocess] 回應長度: {len(response_text)} 字元")
    return {}


print("自定義 node 定義完成。")

In [ ]:
# 使用 LangGraph StateGraph 組裝 workflow
# 在 node 中使用 service.call_llm() 呼叫 LLM

def call_llm_with_analysis(state: dict) -> dict:
    """呼叫 LLM 進行分析。"""
    messages = state.get("messages", [])
    user_text = ""
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            user_text = msg.content
            break
        elif isinstance(msg, tuple) and msg[0] == "user":
            user_text = msg[1]
            break

    response = service.call_llm(
        user_prompt=user_text,
        system_prompt="你是專業的文件分析助手。",
    )
    return {"messages": [AIMessage(content=response.content)]}


graph = StateGraph(MessagesState)
graph.add_node("preprocess", preprocess)
graph.add_node("call_llm", call_llm_with_analysis)
graph.add_node("postprocess", postprocess)

graph.set_entry_point("preprocess")
graph.add_edge("preprocess", "call_llm")
graph.add_edge("call_llm", "postprocess")
graph.add_edge("postprocess", END)

compiled = graph.compile()
print("Workflow 組裝完成：preprocess → call_llm → postprocess → END")

In [ ]:
# 執行自定義 workflow
try:
    final = compiled.invoke({
        "messages": [("user", "LLM 是一種大型語言模型，能夠理解並生成自然語言。它透過大量文本訓練，學會了語言的統計規律。")],
    })
    print("\n=== Workflow 執行完成 ===")
    print(f"訊息記錄（共 {len(final['messages'])} 則）")
    for msg in final["messages"]:
        role = msg.type if hasattr(msg, "type") else msg.get("role", "?")
        content = msg.content if hasattr(msg, "content") else msg.get("content", "")
        print(f"  [{role}] {content[:80]}...")
except Exception as e:
    print(f"[警告] Workflow 執行失敗：{e}")

## 範例三：帶條件分支的 Workflow

使用 `add_conditional_edges` 根據 state 動態決定下一個節點。

自行擴展 `MessagesState` 加入所需欄位：

In [ ]:
from typing import Any


# 擴展 MessagesState 加入自定義欄位
class TaskState(MessagesState):
    task_type: str = "analyze"
    result_prefix: str = ""


def router(state: dict) -> str:
    """根據 task_type 決定路由。"""
    task = state.get("task_type", "analyze")
    if task == "summarize":
        return "summarize"
    return "analyze"


def summarize_node(state: dict) -> dict:
    print("[summarize] 執行摘要任務")
    return {"result_prefix": "（摘要模式）"}


def analyze_node(state: dict) -> dict:
    print("[analyze] 執行分析任務")
    return {"result_prefix": "（分析模式）"}


def call_llm_simple(state: dict) -> dict:
    """簡單 LLM 呼叫 node。"""
    messages = state.get("messages", [])
    user_text = ""
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            user_text = msg.content
            break
        elif isinstance(msg, tuple) and msg[0] == "user":
            user_text = msg[1]
            break

    response = service.call_llm(user_prompt=user_text)
    return {"messages": [AIMessage(content=response.content)]}


# 建構帶條件分支的 workflow
g = StateGraph(TaskState)
g.add_node("call_llm", call_llm_simple)
g.add_node("summarize", summarize_node)
g.add_node("analyze", analyze_node)

g.set_entry_point("call_llm")
g.add_conditional_edges("call_llm", router, {"summarize": "summarize", "analyze": "analyze"})
g.add_edge("summarize", END)
g.add_edge("analyze", END)

conditional_graph = g.compile()

# 測試不同路由
for task in ["summarize", "analyze"]:
    try:
        result = conditional_graph.invoke({
            "messages": [("user", "Hello")],
            "task_type": task,
        })
        print(f"  Task={task} → prefix: {result.get('result_prefix', '')}")
    except Exception as e:
        print(f"  Task={task} → 失敗: {e}")

## 小結

| 元件 | 說明 |
|------|------|
| `LLMService` | 統一 LLM 呼叫入口，在 node 中使用 `service.call_llm()` |
| `MessagesState` | LangGraph 內建 state（只有 messages），可自行擴展 |
| `StateGraph` | LangGraph 原生 API，直接建構 workflow |
| `add_conditional_edges()` | 根據 state 動態路由 |

### MLflow Tracing

- `mlflow.litellm.autolog()` 自動追蹤所有 LiteLLM 呼叫
- 啟動 `mlflow ui --port 5000` 查看完整 trace

### 後續步驟

- `03_evaluation.ipynb`：使用 MLflow GenAI evaluate 進行評估
- `04_prompt_management.ipynb`：Prompt 註冊、版本管理與自動優化